# Home Credit：表内聚合前的数据排查

每次查看一张逻辑表，帮助团队判断表类型，并讨论缺失字段是否有保留价值。只需修改 `TABLE_NAME`，然后运行全部单元格。

## 1. 选择一张表

In [1]:
TABLE_NAME = "static_cb_0"
SAMPLE_ROWS_PER_SHARD = 100
DISPLAY_ROWS = 10

# 可选：base, static_0, static_cb_0, applprev_1, applprev_2,
# credit_bureau_a_1, credit_bureau_a_2, credit_bureau_b_1, credit_bureau_b_2,
# debitcard_1, deposit_1, other_1, person_1, person_2,
# tax_registry_a_1, tax_registry_b_1, tax_registry_c_1

## 2. 环境和逻辑表配置

In [2]:
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

DATA_ROOT = Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability")
TRAIN_ROOT = DATA_ROOT / "parquet_files" / "train"
TABLE_PATTERNS = {
    "base": "train_base.parquet",
    "static_0": "train_static_0_*.parquet",
    "static_cb_0": "train_static_cb_0.parquet",
    "applprev_1": "train_applprev_1_*.parquet",
    "applprev_2": "train_applprev_2.parquet",
    "credit_bureau_a_1": "train_credit_bureau_a_1_*.parquet",
    "credit_bureau_a_2": "train_credit_bureau_a_2_*.parquet",
    "credit_bureau_b_1": "train_credit_bureau_b_1.parquet",
    "credit_bureau_b_2": "train_credit_bureau_b_2.parquet",
    "debitcard_1": "train_debitcard_1.parquet",
    "deposit_1": "train_deposit_1.parquet",
    "other_1": "train_other_1.parquet",
    "person_1": "train_person_1.parquet",
    "person_2": "train_person_2.parquet",
    "tax_registry_a_1": "train_tax_registry_a_1.parquet",
    "tax_registry_b_1": "train_tax_registry_b_1.parquet",
    "tax_registry_c_1": "train_tax_registry_c_1.parquet",
}

assert TABLE_NAME in TABLE_PATTERNS, f"未知表名：{TABLE_NAME}"
SHARDS = tuple(sorted(TRAIN_ROOT.glob(TABLE_PATTERNS[TABLE_NAME])))
assert SHARDS, f"没有匹配到文件：{TABLE_PATTERNS[TABLE_NAME]}"

lf = pl.scan_parquet([str(path) for path in SHARDS])
schema = dict(lf.collect_schema())
keys = [c for c in ["case_id", "num_group1", "num_group2"] if c in schema]
print("当前表：", TABLE_NAME)

当前表： static_cb_0


## 3. 表和 shard 信息

这里只读取 Parquet metadata，不加载完整数据。

In [3]:
shard_rows, shard_schemas = [], []
for path in SHARDS:
    meta = pq.ParquetFile(path).metadata
    file_schema = dict(pl.read_parquet_schema(path))
    shard_schemas.append(file_schema)
    shard_rows.append({
        "shard": path.name,
        "rows": meta.num_rows,
        "columns": len(file_schema),
        "size_mb": round(path.stat().st_size / 1024**2, 2),
    })

shard_info = pd.DataFrame(shard_rows)
N_ROWS = int(shard_info["rows"].sum())
schema_consistent = all(s == shard_schemas[0] for s in shard_schemas[1:])
display(shard_info)
display(pd.DataFrame([{
    "table": TABLE_NAME, "n_shards": len(SHARDS), "n_rows": N_ROWS,
    "n_columns": len(schema), "keys": " + ".join(keys),
    "schema_consistent": schema_consistent,
}]))
assert schema_consistent, "不同 shard 的 schema 不一致，需要单独检查。"

,shard,rows,columns,size_mb
0,train_static_cb_0.parquet,1500476,53,29.23


,table,n_shards,n_rows,n_columns,keys,schema_consistent
0,static_cb_0,1,1500476,53,case_id,True


## 4. 字段名、dtype 和字段说明

In [4]:
feature_description = {}
feature_path = DATA_ROOT / "feature_definitions.csv"
if feature_path.exists():
    feature_def = pd.read_csv(feature_path)
    if {"Variable", "Description"}.issubset(feature_def.columns):
        feature_description = dict(zip(feature_def["Variable"], feature_def["Description"]))

def column_role(name, dtype):
    if name in keys: return "key"
    if dtype.is_temporal(): return "date/time"
    if dtype.is_numeric(): return "numeric"
    if str(dtype) in {"String", "Utf8", "Categorical", "Enum", "Boolean"}: return "categorical"
    return "other"

columns = pd.DataFrame([
    {"column": name, "dtype": str(dtype), "role": column_role(name, dtype),
     "description": feature_description.get(name)}
    for name, dtype in schema.items()
])
display(columns.head(60))


,column,dtype,role,description
0,case_id,Int64,key,None
1,assignmentdate_238D,String,categorical,Tax authority data - date of assignment.
2,assignmentdate_4527235D,String,categorical,Tax authority data - Date of assignment.
3,assignmentdate_4955616D,String,categorical,Tax authority assignment date.
4,birthdate_574D,String,categorical,Client's date of birth (credit bureau data).
5,contractssum_5085716L,Float64,numeric,Total sum of values of contracts retrieved from external credit bureau.
6,dateofbirth_337D,String,categorical,Client's date of birth.
7,dateofbirth_342D,String,categorical,Client's date of birth.
8,days120_123L,Float64,numeric,Number of credit bureau queries for the last 120 days.
9,days180_256L,Float64,numeric,Number of credit bureau queries for last 180 days.


## 5. Sample rows

每个 shard 只取少量开头记录，用于理解一行代表什么。

In [5]:
samples = pl.concat([
    pl.read_parquet(path, n_rows=SAMPLE_ROWS_PER_SHARD)
      .with_columns(pl.lit(path.name).alias("__shard"))
    for path in SHARDS
])
display(samples.head(60).to_pandas())

,case_id,assignmentdate_238D,assignmentdate_4527235D,assignmentdate_4955616D,birthdate_574D,contractssum_5085716L,dateofbirth_337D,dateofbirth_342D,days120_123L,days180_256L,days30_165L,days360_512L,days90_310L,description_5085714M,education_1103M,education_88M,firstquarter_103L,for3years_128L,for3years_504L,for3years_584L,formonth_118L,formonth_206L,formonth_535L,forquarter_1017L,forquarter_462L,forquarter_634L,fortoday_1092L,forweek_1077L,forweek_528L,forweek_601L,foryear_618L,foryear_818L,foryear_850L,fourthquarter_440L,maritalst_385M,maritalst_893M,numberofqueries_373L,pmtaverage_3A,pmtaverage_4527227A,pmtaverage_4955615A,pmtcount_4527229L,pmtcount_4955617L,pmtcount_693L,pmtscount_423L,pmtssum_45A,requesttype_4525192L,responsedate_1012D,responsedate_4527233D,responsedate_4917613D,riskassesment_302T,riskassesment_940T,secondquarter_766L,thirdquarter_1082L,__shard
0,357,None,None,None,1988-04-01,NaN,None,None,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,6301.4000,None,2019-01-25,None,None,None,NaN,NaN,NaN,train_static_cb_0.parquet
1,381,None,None,None,1973-11-01,NaN,None,None,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,4019.6000,None,2019-01-25,None,None,None,NaN,NaN,NaN,train_static_cb_0.parquet
2,388,None,None,None,1989-04-01,NaN,1989-04-01,None,6.0,8.0,2.0,10.0,4.0,a55475b1,a55475b1,a55475b1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,a55475b1,a55475b1,10.0,NaN,NaN,NaN,NaN,NaN,NaN,6.0,14548.0000,None,2019-01-28,None,None,None,NaN,3.0,5.0,train_static_cb_0.parquet
3,405,None,None,None,1974-03-01,NaN,1974-03-01,None,0.0,0.0,0.0,1.0,0.0,a55475b1,a55475b1,a55475b1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,a55475b1,a55475b1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,6.0,10498.2400,None,2019-01-21,None,None,None,NaN,2.0,0.0,train_static_cb_0.parquet
4,409,None,None,None,1993-06-01,NaN,1993-06-01,None,2.0,3.0,0.0,3.0,1.0,a55475b1,717ddd49,a55475b1,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,a7fcb6e5,a55475b1,3.0,NaN,NaN,NaN,NaN,NaN,NaN,7.0,6344.8804,None,2019-01-21,None,None,None,NaN,0.0,4.0,train_static_cb_0.parquet
5,410,None,None,None,1978-06-01,NaN,1978-06-01,None,2.0,2.0,2.0,3.0,2.0,a55475b1,6b2ae0fa,a55475b1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3439d993,a55475b1,3.0,NaN,NaN,NaN,NaN,NaN,NaN,12.0,7200.0000,None,2019-01-22,None,None,None,NaN,3.0,6.0,train_static_cb_0.parquet
6,411,2017-07-06,None,None,1959-07-01,NaN,1959-07-01,None,9.0,9.0,4.0,20.0,7.0,a55475b1,a55475b1,a55475b1,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.0,a55475b1,a55475b1,20.0,5413.3335,NaN,NaN,NaN,NaN,5.0,NaN,NaN,None,2019-01-22,None,None,None,NaN,9.0,2.0,train_static_cb_0.parquet
7,412,None,None,None,1969-01-01,NaN,None,None,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,6284.0000,None,2019-01-22,None,None,None,NaN,NaN,NaN,train_static_cb_0.parquet
8,413,2013-12-25,None,None,1955-12-01,NaN,None,None,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,a55475b1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,a55475b1,a55475b1,NaN,6828.3667,NaN,NaN,NaN,NaN,6.0,NaN,NaN,None,2019-01-22,None,None,None,NaN,NaN,NaN,train_static_cb_0.parquet
9,414,2014-07-30,None,None,1961-06-01,NaN,1961-06-01,None,2.0,3.0,2.0,5.0,2.0,a55475b1,a55475b1,a55475b1,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,3439d993,a55475b1,5.0,5604.8667,NaN,NaN,NaN,NaN,6.0,NaN,NaN,None,2019-01-22,None,None,None,NaN,2.0,1.0,train_static_cb_0.parquet


## 6. 缺失、unique、范围和 flags

缺失值来自全表精确扫描，unique 使用近似值。top values 来自小样本。flags 只用于讨论，不代表应直接删除字段。

In [6]:
exprs = []
for i, (name, dtype) in enumerate(schema.items()):
    exprs.extend([
        pl.col(name).null_count().alias(f"null_{i}"),
        pl.col(name).drop_nulls().approx_n_unique().alias(f"unique_{i}"),
    ])
    if dtype.is_numeric() or dtype.is_temporal():
        exprs.extend([pl.col(name).min().alias(f"min_{i}"), pl.col(name).max().alias(f"max_{i}")])

stats = lf.select(exprs).collect(engine="streaming").row(0, named=True)
profile_rows = []

for i, (name, dtype) in enumerate(schema.items()):
    null_count = int(stats[f"null_{i}"] or 0)
    non_null = N_ROWS - null_count
    n_unique = int(stats[f"unique_{i}"] or 0)
    null_rate = null_count / N_ROWS if N_ROWS else np.nan
    unique_rate = n_unique / non_null if non_null else np.nan

    values = samples.get_column(name).drop_nulls().to_list()
    top_values = Counter(map(repr, values)).most_common(5)
    top1_rate = top_values[0][1] / len(values) if values else np.nan

    flags = []
    if null_rate == 1: flags.append("all_missing")
    elif null_rate >= 0.99: flags.append("missing>=99%")
    if n_unique <= 1: flags.append("constant_candidate")
    elif top1_rate >= 0.99: flags.append("near_constant_in_sample")
    if name not in keys and pd.notna(unique_rate) and unique_rate >= 0.98: flags.append("id_like")

    profile_rows.append({
        "column": name, "dtype": str(dtype), "role": column_role(name, dtype),
        "description": feature_description.get(name), "null_count": null_count,
        "null_rate": null_rate, "approx_n_unique": n_unique,
        "approx_unique_rate": unique_rate, "min": stats.get(f"min_{i}"),
        "max": stats.get(f"max_{i}"), "sample_top_values": top_values,
        "flags": " | ".join(flags),
    })

column_profile = pd.DataFrame(profile_rows)
display(column_profile)

,column,dtype,role,description,null_count,null_rate,approx_n_unique,approx_unique_rate,min,max,sample_top_values,flags
0,case_id,Int64,key,None,0,0.000000,1472623,0.981437,357.000000,2.703454e+06,"[(357, 1), (381, 1), (388, 1), (405, 1), (409, 1)]",
1,assignmentdate_238D,String,categorical,Tax authority data - date of assignment.,1363480,0.908698,8074,0.058936,NaN,NaN,"[('2017-07-06', 1), ('2013-12-25', 1), ('2014-07-30', 1), ('2018-09-21', 1), ('2017-02-25', 1)]",
2,assignmentdate_4527235D,String,categorical,Tax authority data - Date of assignment.,1385498,0.923372,389,0.003383,NaN,NaN,[],
3,assignmentdate_4955616D,String,categorical,Tax authority assignment date.,1428843,0.952260,6784,0.094705,NaN,NaN,[],
4,birthdate_574D,String,categorical,Client's date of birth (credit bureau data).,892605,0.594881,671,0.001104,NaN,NaN,"[('1993-06-01', 2), ('1955-09-01', 2), ('1971-07-01', 2), ('1955-10-01', 2), ('1949-09-01', 2)]",
5,contractssum_5085716L,Float64,numeric,Total sum of values of contracts retrieved from external credit bureau.,1343147,0.895147,139620,0.887440,0.000000,3.129676e+07,[],
6,dateofbirth_337D,String,categorical,Client's date of birth.,114785,0.076499,710,0.000512,NaN,NaN,"[('1993-06-01', 2), ('1955-09-01', 2), ('1971-07-01', 2), ('1952-01-01', 2), ('1989-04-01', 1)]",
7,dateofbirth_342D,String,categorical,Client's date of birth.,1463976,0.975674,671,0.018384,NaN,NaN,"[('1953-07-01', 1)]",near_constant_in_sample
8,days120_123L,Float64,numeric,Number of credit bureau queries for the last 120 days.,114785,0.076499,36,0.000026,0.000000,1.090000e+02,"[(0.0, 21), (2.0, 15), (1.0, 11), (5.0, 5), (3.0, 4)]",
9,days180_256L,Float64,numeric,Number of credit bureau queries for last 180 days.,114785,0.076499,52,0.000038,0.000000,1.100000e+02,"[(0.0, 17), (1.0, 12), (3.0, 10), (2.0, 10), (5.0, 4)]",


### 6.1 筛选全部缺失或非缺失值全部为 0 的数值列。

In [7]:
# 筛选全部缺失或非缺失值全部为 0 的数值列
nan_or_zero_cols = column_profile[
    (column_profile["role"] == "numeric")
    & (
        (column_profile["null_rate"] == 1)
        |
        (
            (column_profile["min"] == 0)
            & (column_profile["max"] == 0)
        )
    )
]

### 6.2 集中显示高缺失、常数候选、近常数和 ID-like 字段。结合字段含义讨论是否保留

In [8]:
display(nan_or_zero_cols)
display(
    column_profile[column_profile["flags"] != ""]
    .sort_values(["null_rate", "column"], ascending=[False, True])
)

,column,dtype,role,description,null_count,null_rate,approx_n_unique,approx_unique_rate,min,max,sample_top_values,flags
20,formonth_118L,Float64,numeric,Number of rejections in a month.,1463962,0.975665,1,0.000027,0.0,0.0,"[(0.0, 1)]",constant_candidate
24,forquarter_462L,Float64,numeric,Number of credit applications that were rejected in the last quarter.,1463962,0.975665,1,0.000027,0.0,0.0,"[(0.0, 1)]",constant_candidate
29,forweek_601L,Float64,numeric,Number of rejected applications in the last week.,1463962,0.975665,1,0.000027,0.0,0.0,"[(0.0, 1)]",constant_candidate


,column,dtype,role,description,null_count,null_rate,approx_n_unique,approx_unique_rate,min,max,sample_top_values,flags
7,dateofbirth_342D,String,categorical,Client's date of birth.,1463976,0.975674,671,0.018384,NaN,NaN,"[('1953-07-01', 1)]",near_constant_in_sample
17,for3years_128L,Float64,numeric,Number of rejected applications in the past 3 years.,1463962,0.975665,2,0.000055,0.000000,1.000000,"[(0.0, 1)]",near_constant_in_sample
18,for3years_504L,Float64,numeric,Client's credit history data over the last three years.,1463962,0.975665,47,0.001287,0.000000,57.000000,"[(9.0, 1)]",near_constant_in_sample
19,for3years_584L,Float64,numeric,Number of cancellations in the last 3 years.,1463962,0.975665,3,0.000082,0.000000,2.000000,"[(0.0, 1)]",near_constant_in_sample
20,formonth_118L,Float64,numeric,Number of rejections in a month.,1463962,0.975665,1,0.000027,0.000000,0.000000,"[(0.0, 1)]",constant_candidate
21,formonth_206L,Float64,numeric,Number of cancelations in the previous month.,1463962,0.975665,2,0.000055,0.000000,1.000000,"[(0.0, 1)]",near_constant_in_sample
22,formonth_535L,Float64,numeric,Credit history for the last month.,1463962,0.975665,8,0.000219,0.000000,11.000000,"[(0.0, 1)]",near_constant_in_sample
23,forquarter_1017L,Float64,numeric,Number of cancellations recorded in the credit bureau in the last quarter.,1463962,0.975665,3,0.000082,0.000000,2.000000,"[(0.0, 1)]",near_constant_in_sample
24,forquarter_462L,Float64,numeric,Number of credit applications that were rejected in the last quarter.,1463962,0.975665,1,0.000027,0.000000,0.000000,"[(0.0, 1)]",constant_candidate
25,forquarter_634L,Float64,numeric,Credit history for the last quarter.,1463962,0.975665,13,0.000356,0.000000,19.000000,"[(4.0, 1)]",near_constant_in_sample


## 7. 记录粒度检查

统计每个 `case_id` 对应的行数，用于初步区分：

- `static/entity`：一个 case 一行
- `repeated entity`：一个 case 对应多个实体
- `history/event`：一个 case 对应多期记录或多个事件

本节只能判断是否存在重复记录。`repeated entity` 和 `history/event` 还需要结合时间结构及字段含义进一步区分。

In [9]:
if "case_id" in schema:
    case_counts = lf.select("case_id").group_by("case_id").len().rename({"len": "rows_per_case"})
    repetition = case_counts.select(
        pl.len().alias("n_cases"),
        pl.col("rows_per_case").mean().alias("mean"),
        pl.col("rows_per_case").median().alias("p50"),
        pl.col("rows_per_case").quantile(0.90).alias("p90"),
        pl.col("rows_per_case").quantile(0.95).alias("p95"),
        pl.col("rows_per_case").quantile(0.99).alias("p99"),
        pl.col("rows_per_case").max().alias("max"),
        (pl.col("rows_per_case") == 1).cast(pl.Float64).mean().alias("one_row_case_rate"),
    ).collect(engine="streaming")
    display(repetition.to_pandas())
else:
    print("没有 case_id，跳过。")

,n_cases,mean,p50,p90,p95,p99,max,one_row_case_rate
0,1500476,1.0,1.0,1.0,1.0,1.0,1,1.0


## 8. 表类型与时间结构判断

## 8.1 日期字段候选

列出可能的日期字段，为后续选择“主要观测时间”做准备。

日期字段可能代表：

- 记录发生时间
- 数据观测时间
- 合同开始或结束时间
- 出生日期
- 申请或响应日期

并不是所有日期字段都能作为 history/event 表的时间轴。

In [10]:
date_mask = (
    column_profile["role"].eq("date/time")
    | column_profile["column"].str.contains(
        "date", case=False, na=False
    )
    | column_profile["column"].str.endswith("D")
)

date_candidates = column_profile[
    date_mask
    & ~column_profile["column"].isin(keys)
].copy()

display(date_candidates[
    [
        "column",
        "dtype",
        "description",
        "null_rate",
        "approx_n_unique",
    ]
])

,column,dtype,description,null_rate,approx_n_unique
1,assignmentdate_238D,String,Tax authority data - date of assignment.,0.908698,8074
2,assignmentdate_4527235D,String,Tax authority data - Date of assignment.,0.923372,389
3,assignmentdate_4955616D,String,Tax authority assignment date.,0.952260,6784
4,birthdate_574D,String,Client's date of birth (credit bureau data).,0.594881,671
6,dateofbirth_337D,String,Client's date of birth.,0.076499,710
7,dateofbirth_342D,String,Client's date of birth.,0.975674,671
46,responsedate_1012D,String,Tax authority's response date.,0.520152,273
47,responsedate_4527233D,String,Tax authority's response date.,0.559922,398
48,responsedate_4917613D,String,Tax authority's response date.,0.850106,212


## 8.2 选择主要观测时间

结合字段说明和 sample rows，选择最能代表“一条记录发生或被观测时间”的字段。

以下日期通常不能直接作为历史时间轴：

- 出生日期
- 合同开始或结束日期
- 客户注册日期
- 只描述实体属性的日期

如果当前表是 static 表，或者没有合适的观测时间，将 `TIME_COL` 保持为 `None`。

In [11]:
# 示例：
# TIME_COL = "deductiondate_4917603D"

TIME_COL = None

print("当前选择的主要观测时间：", TIME_COL)

当前选择的主要观测时间： None


## 8.3 时间结构判断

如果当前表存在主要观测时间，本节判断其时间结构：

- `monthly`：接近连续月份记录
- `annual`：接近年度记录
- `irregular event`：事件发生时间不规则
- `sparse/single-period`：大部分 case 只有一期记录

月度表可以考虑最近 3/6/12 个月；年度表应使用最近 1/2/3 年，不能直接套用月度窗口。

In [12]:
rep = repetition.row(0, named=True)

time_pattern = None
time_result = pd.DataFrame()

if rep["max"] == 1:

    print(
        "当前表每个 case 只有一行，"
        "不需要分析历史时间结构。"
    )

elif TIME_COL is None:

    print(
        "TIME_COL=None，暂不分析时间结构。"
    )

elif TIME_COL not in schema:

    print(f"{TIME_COL} 不在当前表中。")

else:

    dtype = schema[TIME_COL]

    if dtype.is_temporal():

        time_expr = (
            pl.col(TIME_COL)
            .cast(pl.Date, strict=False)
            .alias("_time")
        )

    elif str(dtype) in {
        "String",
        "Utf8",
        "Categorical",
        "Enum",
    }:

        time_expr = (
            pl.col(TIME_COL)
            .cast(pl.String)
            .str.to_date(strict=False)
            .alias("_time")
        )

    else:

        raise ValueError(
            "该字段无法自动转换成日期，请检查日期编码。"
        )

    non_null_count = samples.select(
        pl.col(TIME_COL).is_not_null().sum()
    ).item()

    parsed = (
        samples
        .select("case_id", time_expr)
        .drop_nulls("_time")
    )

    parse_rate = (
        parsed.height / non_null_count
        if non_null_count else 0
    )

    if parsed.is_empty():

        print("sample 中没有成功解析的日期。")

    else:

        # 每个 case 有多少个不同时间点
        per_case = (
            parsed
            .group_by("case_id")
            .agg(
                pl.col("_time")
                .n_unique()
                .alias("n_periods")
            )
        )

        periods = per_case.get_column("n_periods")

        multi_period_rate = float(
            (periods > 1).mean()
        )

        # 计算相邻时间间隔
        time_points = (
            parsed
            .unique(["case_id", "_time"])
            .sort(["case_id", "_time"])
            .with_columns(
                (
                    pl.col("_time").dt.year() * 12
                    + pl.col("_time").dt.month()
                ).alias("_month_index")
            )
            .with_columns(
                pl.col("_time")
                .diff()
                .over("case_id")
                .dt.total_days()
                .alias("gap_days"),

                pl.col("_month_index")
                .diff()
                .over("case_id")
                .alias("gap_months"),
            )
        )

        gaps = time_points.filter(
            pl.col("gap_days").is_not_null()
        )

        if gaps.height > 0:

            gap_days = gaps.get_column("gap_days")
            gap_months = gaps.get_column("gap_months")

            monthly_gap_rate = float(
                (gap_months == 1).mean()
            )

            remainder = gap_months % 12

            annual_mask = (
                (gap_months >= 11)
                & (
                    (remainder <= 1)
                    | (remainder >= 11)
                )
            )

            annual_gap_rate = float(
                annual_mask.mean()
            )

            median_gap_days = gap_days.median()

        else:

            monthly_gap_rate = 0
            annual_gap_rate = 0
            median_gap_days = None

        if (
            periods.median() <= 1
            or multi_period_rate < 0.20
        ):
            time_pattern = "sparse/single-period"

        elif monthly_gap_rate >= 0.50:
            time_pattern = "monthly"

        elif annual_gap_rate >= 0.50:
            time_pattern = "annual"

        else:
            time_pattern = "irregular event"

        time_result = pd.DataFrame([{
            "time_column": TIME_COL,
            "parse_rate": parse_rate,
            "sample_cases": per_case.height,
            "median_periods_per_case": periods.median(),
            "max_periods_per_case": periods.max(),
            "multi_period_case_rate": multi_period_rate,
            "median_gap_days": median_gap_days,
            "monthly_gap_rate": monthly_gap_rate,
            "annual_gap_rate": annual_gap_rate,
            "time_pattern": time_pattern,
        }])

        display(time_result)

当前表每个 case 只有一行，不需要分析历史时间结构。


## 8.4 Numeric 字段语义

Numeric 字段不能统一使用相同的聚合函数，需要结合字段含义判断：

- `flow`：交易金额、支付金额，适合 sum、mean、max
- `stock`：余额、当前状态，适合 latest、mean、max，不适合 sum
- `cumulative`：累计金额或次数，适合 latest、max、latest-first
- `rate`：比率，适合 mean、latest、min、max
- `identifier`：编号，只适合 count 或 nunique

本节列出数值字段，供团队人工判断字段语义。

In [13]:
numeric_review = column_profile[
    (column_profile["role"] == "numeric")
    & ~column_profile["column"].isin(keys)
][
    [
        "column",
        "description",
        "null_rate",
        "approx_n_unique",
        "min",
        "max",
        "flags",
    ]
].copy()

display(numeric_review)

,column,description,null_rate,approx_n_unique,min,max,flags
5,contractssum_5085716L,Total sum of values of contracts retrieved from external credit bureau.,0.895147,139620,0.000000,3.129676e+07,
8,days120_123L,Number of credit bureau queries for the last 120 days.,0.076499,36,0.000000,1.090000e+02,
9,days180_256L,Number of credit bureau queries for last 180 days.,0.076499,52,0.000000,1.100000e+02,
10,days30_165L,Number of credit bureau queries for the last 30 days.,0.076499,21,0.000000,2.200000e+01,
11,days360_512L,Number of Credit Bureau queries for last 360 days.,0.076499,86,0.000000,1.150000e+02,
12,days90_310L,Number of credit bureau queries for the last 90 days.,0.076499,32,0.000000,4.100000e+01,
16,firstquarter_103L,Number of results obtained from credit bureau in the first quarter.,0.076499,56,0.000000,7.600000e+01,
17,for3years_128L,Number of rejected applications in the past 3 years.,0.975665,2,0.000000,1.000000e+00,near_constant_in_sample
18,for3years_504L,Client's credit history data over the last three years.,0.975665,47,0.000000,5.700000e+01,near_constant_in_sample
19,for3years_584L,Number of cancellations in the last 3 years.,0.975665,3,0.000000,2.000000e+00,near_constant_in_sample


## 8.5 聚合方向建议

聚合规则由三部分决定：

1. 第 7 节的记录粒度决定聚合层级。
2. 时间结构决定使用月份、年份还是事件聚合。
3. Numeric 字段语义决定使用 sum、latest、change 或其他统计。

本节生成当前表的总体聚合建议，最终结果仍需结合字段说明人工确认。

In [14]:
# 表类型候选
if "num_group2" in schema:

    table_type_candidate = "nested/detail"

    structure_strategy = (
        "先按 case_id + num_group1 聚合，"
        "再按 case_id 聚合。"
    )

elif rep["max"] == 1:

    table_type_candidate = "static/entity"

    structure_strategy = (
        "不需要 groupby，检查 case_id 唯一后直接 join。"
    )

elif time_pattern in {"monthly", "annual"}:

    table_type_candidate = "history/event candidate"

    structure_strategy = (
        "按 case_id 聚合，并保留时间窗口特征。"
    )

elif time_pattern == "irregular event":

    table_type_candidate = (
        "history/event 或 repeated entity"
    )

    structure_strategy = (
        "按 case_id 聚合，但需要确认日期是事件时间还是实体属性。"
    )

else:

    table_type_candidate = "repeated entity candidate"

    structure_strategy = (
        "按 case_id 聚合，重点统计实体数量和实体属性。"
    )


# 时间聚合建议
if time_pattern == "monthly":

    time_strategy = (
        "latest、最近3/6/12个月、trend、"
        "volatility、连续异常月份。"
    )

elif time_pattern == "annual":

    time_strategy = (
        "latest、最近1/2/3年、年度均值、"
        "YoY change、active_years。"
    )

elif time_pattern == "irregular event":

    time_strategy = (
        "event_count、recency、时间间隔、"
        "amount sum/max；不使用连续月份。"
    )

elif time_pattern == "sparse/single-period":

    time_strategy = (
        "has_record、record_count、latest、recency；"
        "不建议计算趋势。"
    )

else:

    time_strategy = (
        "没有确定的历史时间轴，暂不使用时间窗口。"
    )


aggregation_suggestion = pd.DataFrame([{
    "table": TABLE_NAME,
    "table_type_candidate": table_type_candidate,
    "time_column": TIME_COL,
    "time_pattern": time_pattern,
    "structure_strategy": structure_strategy,
    "time_strategy": time_strategy,
    "missing_strategy": (
        "保留 record_count、valid_count、"
        "missing_count 和 missing_rate。"
    ),
}])

display(aggregation_suggestion)

,table,table_type_candidate,time_column,time_pattern,structure_strategy,time_strategy,missing_strategy
0,static_cb_0,static/entity,None,None,不需要 groupby，检查 case_id 唯一后直接 join。,没有确定的历史时间轴，暂不使用时间窗口。,保留 record_count、valid_count、missing_count 和 missing_rate。


## 9. 团队判断

讨论以下问题：

1. 一行代表客户、合同、账户、事件、月度记录还是明细？
2. `case_id` 为什么重复？`num_group1` / `num_group2` 表示什么层级？
3. 是否存在代表记录时间的字段？
4. 高缺失字段是否有业务意义，缺失本身是否有信息？
5. 常数、近常数和 ID-like 字段是否有保留价值？

最终类型：`static/entity`、`repeated entity`、`history/event`、`nested/detail` 或 `other`。

### 人工记录

- 表类型：
- 一行的含义：
- 实体层级：
- 主要时间字段：
- 建议保留/删除的字段：
- 后续聚合层级：
- 备注：